# Task 1.3 — Baseline Comparison and the Role of the 0.95 Exponent

**Paper:** Goorha, S. & Ungar, L. (2010). *Discovery of Significant Emerging Trends.* ACM SIGKDD.

---

## The Baseline: Simple Frequency Tracking

The most natural baseline for trend detection is **simple frequency counting** — rank phrases by how often they appear in the corpus. This approach is trivial to implement:

$$\text{Score}_{\text{baseline}} = \text{count}$$

However, simple frequency tracking suffers from a critical flaw: **common "noise" words dominate the rankings**. Stop-words like "the", "and", "is", and generic phrases like "in the" or "of the" will always rank at the top because they appear far more frequently than any meaningful trend. The signal (emerging trends) is drowned out by the noise (ubiquitous language).

## How the 0.95 Exponent Overcomes This Limitation

The Discovery method replaces simple frequency with a **power-law score**:

$$\text{Score} = \frac{\text{count}}{\text{total}^{0.95}}$$

The key insight is in the **exponent value of 0.95**:

### Why not α = 0.0 (Pure Frequency)?
With α = 0.0, the denominator becomes `total^0 = 1`, so `Score = count`. This is identical to the baseline — noise words dominate.

### Why not α = 1.0 (Pure Normalisation)?
With α = 1.0, the score becomes `count / total`, which is simply the relative frequency. While this removes the bias towards high-frequency words, it **overcorrects**: any phrase that appears exactly once in a small corpus gets a score of `1/1 = 1.0` (a perfect score). Rare junk phrases and typos would dominate the rankings.

### Why α = 0.95 works
The exponent 0.95 sits in a **sweet spot**:
- It **penalises high-frequency noise** by dividing by a near-linear function of total count
- It **avoids overcorrecting** towards rare phrases because `total^0.95` grows slightly slower than `total^1.0`
- Phrases need to be **both frequent enough** to be meaningful **and specific enough** to not be generic stop-words

In essence, the 0.95 exponent creates a **signal-to-noise filter** that surfaces genuinely emerging trends while suppressing both ubiquitous language noise and rare statistical artefacts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate the effect of different exponents on scoring
phrases = ['the', 'and', 'iPhone 15 release', 'Tesla recall', 'xyz_typo']
counts   = [50000,  40000,  500,                300,            1]
total    = sum(counts)

exponents = [0.0, 0.5, 0.95, 1.0]

print(f"{'Phrase':<22} {'Count':>7}  {'α=0.0':>10}  {'α=0.5':>10}  {'α=0.95':>10}  {'α=1.0':>10}")
print('-' * 78)
for phrase, count in zip(phrases, counts):
    scores = [count / (total ** a) if a > 0 else count for a in exponents]
    print(f"{phrase:<22} {count:>7}  {scores[0]:>10.4f}  {scores[1]:>10.6f}  {scores[2]:>10.6f}  {scores[3]:>10.6f}")

The table above shows how the 0.95 exponent suppresses noise words (`the`, `and`) while still ranking genuine trends (`iPhone 15 release`, `Tesla recall`) above rare junk (`xyz_typo`). This is the core advantage over simple frequency tracking.